# uvipen/Super-mario-bros-PPO-pytorch — Ejecutar modelos pre-entrenados

Este repo usa **PyTorch puro + CNN sobre píxeles** (diferente al de tu docente que usa SB3 + RAM grid).

Tiene modelos pre-entrenados para **31/32 niveles** en la carpeta `trained_models/`.

### Entorno conda requerido (DIFERENTE a mario_env):
```bash
conda create -n mario_uvipen python=3.8
conda activate mario_uvipen
pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
pip install gym-super-mario-bros==7.3.2
pip install opencv-python==4.8.0.76
pip install pyglet==1.5.27
pip install nes-py==8.2.1
```

In [2]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import torch
import torch.nn.functional as F
import numpy as np
import time

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 1.13.1+cu117
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
# ── Verificar que estás en la carpeta correcta del repo ──────────────────────
import os
files = os.listdir('.')
assert 'trained_models' in files, "ERROR: Ejecuta este notebook desde la carpeta raíz del repo uvipen"
assert 'src' in files, "ERROR: No se encuentra la carpeta src/"

models = os.listdir('trained_models')
print(f"Modelos disponibles ({len(models)} total):")
for m in sorted(models)[:10]:
    print(f"  {m}")
if len(models) > 10:
    print(f"  ... y {len(models)-10} más")

Modelos disponibles (31 total):
  ppo_super_mario_bros_1_1
  ppo_super_mario_bros_1_2
  ppo_super_mario_bros_1_3
  ppo_super_mario_bros_1_4
  ppo_super_mario_bros_2_1
  ppo_super_mario_bros_2_2
  ppo_super_mario_bros_2_3
  ppo_super_mario_bros_2_4
  ppo_super_mario_bros_3_1
  ppo_super_mario_bros_3_2
  ... y 21 más


In [4]:
# ── Configuración ─────────────────────────────────────────────────────────────
WORLD       = 6      # Mundo (1-8)
STAGE       = 2      # Nivel (1-4)
ACTION_TYPE = 'simple'  # 'simple', 'right', 'complex'

# Ruta al modelo pre-entrenado
MODEL_PATH = f'trained_models/ppo_super_mario_bros_{WORLD}_{STAGE}'

# Verificar que existe
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f'✅ Modelo encontrado: {MODEL_PATH} ({size_mb:.1f} MB)')
else:
    print(f'❌ Modelo NO encontrado: {MODEL_PATH}')
    print('Modelos disponibles:')
    for m in sorted(os.listdir('trained_models')):
        print(f'  trained_models/{m}')

✅ Modelo encontrado: trained_models/ppo_super_mario_bros_6_2 (2.5 MB)


In [5]:
# ── Importar el entorno y modelo del repo ────────────────────────────────────
from src.env import create_train_env
from src.model import PPO
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT, COMPLEX_MOVEMENT, RIGHT_ONLY

# Seleccionar acciones
if ACTION_TYPE == 'right':
    actions = RIGHT_ONLY
elif ACTION_TYPE == 'simple':
    actions = SIMPLE_MOVEMENT
else:
    actions = COMPLEX_MOVEMENT

print(f'Acciones: {ACTION_TYPE} ({len(actions)} acciones)')
print(f'Nivel: W{WORLD}-{STAGE}')

Acciones: simple (7 acciones)
Nivel: W6-2


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
# ── Crear entorno manualmente (sin depender de create_train_env) ─────────────
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from src.env import CustomReward, CustomSkipFrame

# Crear cadena manualmente para garantizar 4 frames
env = gym_super_mario_bros.make(f'SuperMarioBros-{WORLD}-{STAGE}-v0',
                                 max_episode_steps=None)
# Quitar TimeLimit si gym lo añadió igual
import gym.wrappers.time_limit as _tl
while isinstance(env, _tl.TimeLimit):
    env = env.env

env = JoypadSpace(env, actions)
env = CustomReward(env, world=WORLD, stage=STAGE)
env = CustomSkipFrame(env, skip=4)

print(f'Observation shape: {env.observation_space.shape}')  # debe ser (4, 84, 84)
print(f'Action space: {env.action_space.n}')

# Verificar shape del reset
state_test = env.reset()
print(f'State shape: {state_test.shape}')   # debe ser (1, 4, 84, 84)
print(f'State dtype: {state_test.dtype}')   # debe ser float32

/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-6-2-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(


✅ Entorno creado sin grabación de video
Observation shape: (4, 84, 84)
Action space: 7


In [7]:
# ── Cargar modelo pre-entrenado ───────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = PPO(env.observation_space.shape[0], len(actions))

if torch.cuda.is_available():
    model.load_state_dict(torch.load(MODEL_PATH))
    model.cuda()
else:
    model.load_state_dict(
        torch.load(MODEL_PATH, map_location=lambda storage, loc: storage)
    )

model.eval()
print(f'✅ Modelo cargado en {device}')
print(f'Parámetros: {sum(p.numel() for p in model.parameters()):,}')

✅ Modelo cargado en cuda
Parámetros: 623,368


In [8]:
# ── Jugar con ventana visual ──────────────────────────────────────────────────
# env.render() abre la ventana del emulador NES
# IMPORTANTE: necesitas display (no funciona en servidor sin pantalla)

EPISODES    = 3
FRAME_DELAY = 0.02   # ~50 FPS  (bajar si va muy rápido, subir si va lento)
DETERMINISTIC = True  # True = siempre la misma acción, False = algo de aleatoriedad

scores = []
state_test = env.reset()
print(f"Shape del estado: {state_test.shape}")
print(f"Tipo: {state_test.dtype}")
print(f"Tipo env: {type(env)}")

for ep in range(1, EPISODES + 1):
    state = torch.from_numpy(env.reset()).float()
    done  = False
    score = 0.0
    step  = 0
    completed = False

    print(f'\n═══ Episodio {ep}/{EPISODES} ═══')

    while not done:
        # Render — abre/actualiza la ventana del emulador
        env.render()

        # Inferencia
        if torch.cuda.is_available():
            state = state.cuda()

        with torch.no_grad():
            logits, value = model(state)
            policy = F.softmax(logits, dim=1)

            if DETERMINISTIC:
                action = torch.argmax(policy).item()
            else:
                action = torch.multinomial(policy, 1).item()

        state, reward, done, info = env.step(action)
        state = torch.from_numpy(state).float()
        score += reward
        step  += 1

        # Mostrar info cada 100 pasos
        if step % 100 == 0:
            x_pos = info.get('x_pos', 0)
            print(f'  Step {step:4d} | Score: {score:7.1f} | X: {x_pos} | Vida: {info.get("life", "?")}', end='\r')

        if info.get('flag_get', False):
            completed = True
            print(f'\n  🏁 ¡NIVEL COMPLETADO! Score: {score:.0f}')
            time.sleep(1.5)
            break

        time.sleep(FRAME_DELAY)

    scores.append(score)
    status = '✅ COMPLETADO' if completed else '❌ Murió'
    print(f'\n  {status} | Score final: {score:.0f} | Steps: {step}')

print(f"\n{'='*40}")
print(f'Score medio : {np.mean(scores):.1f}')
print(f'Score máximo: {np.max(scores):.1f}')
print(f'Score mínimo: {np.min(scores):.1f}')

Shape del estado: (1, 84, 84)
Tipo: float64
Tipo env: <class 'src.env.CustomSkipFrame'>

═══ Episodio 1/3 ═══


/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/deynar/miniconda3/envs/mario_uvipen/lib/python3.8/site-packages/gym/utils/passive_env_checker.py:272: UserWarning: WARN: No render modes was declared in the environment (env.metadata['render_modes'] is None or not defined), you may have trouble when calling `.render()`.
  logger.warn(


RuntimeError: Given groups=1, weight of size [32, 4, 3, 3], expected input[1, 1, 84, 84] to have 4 channels, but got 1 channels instead

In [9]:
env.close()
print('Entorno cerrado.')

Entorno cerrado.


## Modelos disponibles en trained_models/

| Mundo-Nivel | Tipo | Notas |
|---|---|---|
| W1-1 a W8-3 | SIMPLE | La mayoría completados |
| W8-4 | — | No resuelto (laberinto) |

Para cambiar de nivel solo modifica `WORLD` y `STAGE` en la celda de configuración.

## Diferencias con el repo de tu docente

| Característica | uvipen (este) | docente (yumouwei) |
|---|---|---|
| Framework | PyTorch puro | Stable Baselines3 |
| Observación | Imagen RGB 84×84 (CNN) | Grid RAM 16×13 (MLP) |
| Formato modelo | `.pt` / state_dict | `.zip` SB3 |
| Acciones | SIMPLE por defecto | SIMPLE/RIGHT/COMPLEX |
| Niveles pre-entrenados | 31/32 | Solo W1-1 |

Los modelos de ambos repos **NO son compatibles entre sí** por las diferencias de arquitectura.